# One Model, Two Tests: Does It Know Tamil Vocabulary, or Tamil Grammar?

A single small **GPT-style decoder** (the same kind of architecture behind tools like ChatGPT, just far smaller and Tamil-only) — no separate "reader" and "writer" halves like the mBART notebooks, just one stream that reads a verse and continues it. It's trained on **two tasks at once, sharing the same weights**:

1. **Verse LM** — just predict the next word of a verse, one token at a time (no commentary involved).
2. **Verse → Urai** — given a verse, predict its commentary word by word.

Once trained, it faces two very different tests:

1. **Test 1 — Urai prediction vs gold:** generate a commentary for a verse and score it against the real, human-written commentary. This asks: *does it know the right words?*
2. **Test 2 — Clause preference:** take a real verse, scramble its word order four different linguistically-motivated ways, and ask the model which version — the original or a scrambled one — looks most like real Tamil. This asks a completely different question: *does it know the right word order?*

The clause parses driving Test 2 come from a fresh batch of 20 verses spread across **all five source texts** (Tholkappiyam Ezhuthadhikaram/Porulathikaram/Sollathikaram, Naaladiyar, Thirukadukam) — broader coverage than the original single-nool sample, so this run's Test 2 result generalises further than before.

## v2 changes (corrected dataset rerun)
- **Data**: loads the corrected corpus from local `data/` (extract `classical_tamil_verse_urai_corpus.zip` there) --- **1,262 pairs, 393 Naaladiyar** (verse #398 explanation fixed). A sanity-gate cell asserts these counts before anything trains.
- **Outputs**: written to local `outputs/` instead of `/kaggle/working/`.
- Seed unchanged (3407) so results are comparable to the v1 (1,261-pair) run.
- **NEW: 90/10 held-out split** --- the joint LM now trains on `train_rows` only, and **Test 1 (urai token-F1) is now evaluated on held-out verses**, fixing the v1 caveat that it scored training data.
- Test 2 (clause/grammar probe) still uses the 19 GPT-4.1-mini-parsed verses wherever they fall (train or held-out) because the parse set is small; the memorisation caveat therefore still applies to Test 2 and must be stated in the write-up.

In [ ]:
import json, math, random, re, zlib
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 3407
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# --- v3: quiet, reproducible execution ---------------------------------------
import os, warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm as _tqdm
_QUIET = os.environ.get("NB_VERBOSE", "0") != "1"
def tqdm(iterable=None, *args, **kwargs):
    """Silent under nbconvert; set NB_VERBOSE=1 to get bars back."""
    kwargs.setdefault("disable", _QUIET)
    kwargs.setdefault("leave", False)
    return _tqdm(iterable, *args, **kwargs)


In [ ]:
def configure_tamil_font():
    candidates = ["Noto Sans Tamil", "Nirmala UI", "Latha", "Arial Unicode MS", "FreeSerif", "DejaVu Sans"]
    installed = {f.name for f in fm.fontManager.ttflist}
    chosen = next((n for n in candidates if n in installed), "DejaVu Sans")
    plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    print("Tamil font:", chosen)

configure_tamil_font()

# --- v3: one plot style for every figure in the ladder ------------------------
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.frameon": False, "legend.fontsize": 8.5,
    "figure.facecolor": "white", "axes.facecolor": "white",
})
Path("outputs").mkdir(exist_ok=True)


In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).replace("\ufeff", " ").replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

# --- v3: minimal special-token set -------------------------------------------
# Removed vs v2, deliberately:
#   * dataset identity tags <naladiyar>/<thirukadukam>/<tholkappiyam>
#     -> they let a model win by recognising WHICH text a pair came from
#        (the source-identification confound behind the 0.78 combined
#        pair-classifier) instead of whether verse and urai actually match.
#   * <lb> line-break token
#     -> tokenize_words only emitted it for text that still had newlines. In
#        the grammar probe the ORIGINAL verse kept its newlines while every
#        scrambled variant was rebuilt as a single line, so the original was
#        scored with (n_lines - 1) extra, highly predictable <lb> tokens that
#        no competitor had. That inflated the 19/19 result. Gone.
PAD, UNK, MASK, BOS, EOS = "<pad>", "<unk>", "<mask>", "<bos>", "<eos>"
VERSE_TAG, URAI_TAG = "<verse>", "<urai>"

SPECIAL_TOKENS = [PAD, UNK, MASK, BOS, EOS, VERSE_TAG, URAI_TAG]

def tokenize_words(text: str):
    """v3: flat word list. Line structure is NOT encoded (see the <lb> note
    above), so a verse and any re-linearised rewrite of it are tokenized on
    exactly equal terms."""
    text = normalize_text(text).lower()
    tokens = []
    for word in text.split():
        cleaned = re.sub(r"[^\w\u0B80-\u0BFF]+", "", word)
        if cleaned: tokens.append(cleaned)
    return tokens

def encode_tokens(tokens, stoi):
    unk = stoi.get(UNK, 1)
    return [stoi.get(t, unk) for t in tokens]

def norm(t):
    return re.sub(r"\s+", " ", normalize_text(t)).strip()

In [ ]:
# Each dataset lists candidate paths (Kaggle first, then local repo fallbacks);
# the first path that exists is used, missing datasets are skipped with a warning.
DATA_SOURCES = [
    ("naladiyar", [
        "data/naladiyar.jsonl",
        "../naaladiyar_tamilvu_verse_explanation.jsonl",
        "naaladiyar_tamilvu_verse_explanation.jsonl",
    ]),
    ("tholkappiyam_eth", [
        "data/sft_data-eth.jsonl",
        "../eluttatikaram_balasundaram.jsonl",
        "eluttatikaram_balasundaram.jsonl",
    ]),
    ("tholkappiyam_por", [
        "data/sft_data-por.jsonl",
    ]),
    ("tholkappiyam_sol", [
        "data/sft_data-sol.jsonl",
        "../balasundaram_sollathikaram_verse_explanation_cleaned.jsonl",
        "balasundaram_sollathikaram_verse_explanation_cleaned.jsonl",
    ]),
    ("thirukadukam", [
        "data/thirukadukam_tamilvu.jsonl",
        "../thirukadukam_tamilvu.jsonl",
        "thirukadukam_tamilvu.jsonl",
    ]),
]

def load_rows(sources):
    rows = []
    for ds, candidates in sources:
        path = next((p for p in candidates if Path(p).exists()), None)
        if path is None:
            print(f"WARNING: no file found for {ds} - skipped")
            continue
        n0 = len(rows)
        with open(path, "r", encoding="utf-8-sig", errors="ignore") as f:
            for line in f:
                if not line.strip(): continue
                obj   = json.loads(line)
                verse = normalize_text(obj.get("verse", ""))
                urai  = normalize_text(obj.get("explanation", obj.get("explaination", obj.get("urai", ""))))
                if verse and urai:
                    rows.append({"dataset": ds, "verse": verse, "urai": urai})
        print(f"{ds:18s} {len(rows)-n0:4d} rows  ({path})")
    return rows

all_rows = load_rows(DATA_SOURCES)
df_all   = pd.DataFrame(all_rows)
print(f"\nTotal rows: {len(all_rows)}")
print(df_all["dataset"].value_counts())

In [ ]:
# v2 sanity gate: the corrected corpus MUST load 1,262 pairs (393 Naaladiyar).
# If this fails, the data/ folder does not contain the corrected files from
# classical_tamil_verse_urai_corpus.zip - fix that before training anything.
from collections import Counter as _Counter
_c = _Counter(r["dataset"] for r in all_rows)
print("per-dataset:", dict(_c), " total:", len(all_rows))
assert len(all_rows) == 1262, f"expected 1,262 rows, got {len(all_rows)}"
assert _c.get("naladiyar") == 393, f"expected 393 Naaladiyar rows, got {_c.get('naladiyar')}"
print("OK: corrected corpus confirmed (1,262 pairs, 393 Naaladiyar)")

In [ ]:
import random
# v2: deterministic 90/10 held-out split (seed 3407).
# Training / SFT use train_rows only; heldout_rows is never trained on.
_split_rng = random.Random(3407)
_order = list(range(len(all_rows)))
_split_rng.shuffle(_order)
_n_hold = max(1, int(0.10 * len(all_rows)))
heldout_rows = [all_rows[i] for i in _order[:_n_hold]]
train_rows   = [all_rows[i] for i in _order[_n_hold:]]
print(f"split: train={len(train_rows)}  heldout={len(heldout_rows)}")

In [ ]:
def build_vocab(rows, min_freq=1):
    counts = Counter()
    for row in rows:
        counts.update(tokenize_words(row["verse"]))
        counts.update(tokenize_words(row["urai"]))
    stoi = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    for tok, freq in counts.items():
        if freq >= min_freq and tok not in stoi:
            stoi[tok] = len(stoi)
    return stoi

vocab = build_vocab(all_rows)
itos  = {v: k for k, v in vocab.items()}
PAD_ID, UNK_ID, BOS_ID, EOS_ID = vocab[PAD], vocab[UNK], vocab[BOS], vocab[EOS]
print(f"Vocab size: {len(vocab)}")

## Architecture and joint training

One causal Transformer (4 layers, 4 heads, 256-dim) — the same single stream of weights is updated by both tasks in the same shuffled batch: verse-only examples teach it to predict verse text; verse→urai examples teach it to predict commentary text (loss is only applied to the commentary tokens there, the verse portion is just context it reads).

**Real result from this run:** both objectives converge together, roughly **8.8 → 2.2** (verse) and **9.3 → 2.2** (urai) per-token loss over 30 epochs on 2,522 combined training sequences (1,262 verses + 1,262 verse-urai pairs) — 17.8M parameters total.

In [ ]:
class CausalBlock(nn.Module):
    def __init__(self, d, nh, ff, dr):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, nh, dropout=dr, batch_first=True)
        self.ff1  = nn.Linear(d, ff); self.ff2 = nn.Linear(ff, d)
        self.n1   = nn.LayerNorm(d);  self.n2  = nn.LayerNorm(d)
        self.drop = nn.Dropout(dr)
    def forward(self, x, cmask):
        a, _ = self.attn(x, x, x, attn_mask=cmask, need_weights=False)
        x = self.n1(x + self.drop(a))
        return self.n2(x + self.drop(self.ff2(self.drop(F.gelu(self.ff1(x))))))

class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, d=256, nh=4, n_layers=4, ff=512, dr=0.1,
                 max_len=512, pad_id=0):
        super().__init__()
        self.d = d
        self.max_len = max_len
        self.emb = nn.Embedding(vocab_size, d, padding_idx=pad_id)
        pe  = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
        self.drop   = nn.Dropout(dr)
        self.layers = nn.ModuleList([CausalBlock(d, nh, ff, dr) for _ in range(n_layers)])
        self.head   = nn.Linear(d, vocab_size)

    def forward(self, ids):                          # ids: [B, T]
        T = ids.size(1)
        x = self.drop(self.emb(ids) * math.sqrt(self.d) + self.pe[:, :T])
        m = torch.triu(torch.ones(T, T, device=ids.device, dtype=torch.bool), diagonal=1)
        for l in self.layers: x = l(x, m)
        return self.head(x)                          # [B, T, V]

def collate_joint(batch):
    L    = max(len(b["ids"]) for b in batch)
    inp  = torch.full((len(batch), L - 1), PAD_ID, dtype=torch.long)
    tgt  = torch.full((len(batch), L - 1), -100,   dtype=torch.long)
    task = torch.tensor([b["task"] for b in batch], dtype=torch.long)  # 0=verse, 1=urai
    for i, b in enumerate(batch):
        ids = b["ids"]; n = len(ids); s = b["loss_start"]
        inp[i, :n-1]    = torch.tensor(ids[:-1])
        tgt[i, s-1:n-1] = torch.tensor(ids[s:])
    return inp, tgt, task

@torch.no_grad()
def sequence_logprob(model, ids, loss_start):
    """Total log-probability (nats) of ids[loss_start:] given the preceding tokens,
    plus the number of scored tokens."""
    model.eval()
    inp  = torch.tensor([ids[:-1]], dtype=torch.long, device=DEVICE)
    logp = F.log_softmax(model(inp)[0], dim=-1)                  # [T-1, V]
    tgt  = torch.tensor(ids[1:], dtype=torch.long, device=DEVICE)
    sel  = logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)[loss_start-1:]
    return float(sel.sum()), int(sel.numel())

In [ ]:
VERSE_PREFIX_LEN = 2   # v3: BOS + VERSE_TAG (dataset tag removed)

class JointDataset(Dataset):
    """Mix of the two objectives: task 0 = verse LM, task 1 = verse->urai."""
    def __init__(self, rows, vocab, max_len=320):
        self.items = []
        for r in rows:
            vt = [BOS, VERSE_TAG] + tokenize_words(r["verse"])
            # objective (a): verse LM
            ids = encode_tokens(vt + [EOS], vocab)[:max_len]
            if len(ids) > VERSE_PREFIX_LEN + 1:
                self.items.append({"ids": ids, "loss_start": VERSE_PREFIX_LEN, "task": 0})
            # objective (b): urai given verse
            ut  = [URAI_TAG] + tokenize_words(r["urai"]) + [EOS]
            ids = encode_tokens(vt + ut, vocab)[:max_len]
            loss_start = len(vt) + 1                 # first urai word (URAI_TAG is given)
            if loss_start < len(ids):
                self.items.append({"ids": ids, "loss_start": loss_start, "task": 1})
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def train_joint(model, dataset, epochs=30, lr=3e-4, batch_size=16):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_joint)
    opt    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist   = {"verse": [], "urai": []}
    for ep in range(1, epochs + 1):
        model.train()
        sums = {0: 0.0, 1: 0.0}; cnts = {0: 0, 1: 0}
        for inp, tgt, task in loader:
            inp, tgt = inp.to(DEVICE), tgt.to(DEVICE)
            opt.zero_grad()
            logits = model(inp)
            tok = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                  tgt.reshape(-1), ignore_index=-100,
                                  reduction="none").reshape(tgt.shape)
            mask = tgt != -100
            loss = tok[mask].mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            for t in (0, 1):
                m = mask[task == t]
                sums[t] += float(tok[task == t][m].sum()); cnts[t] += int(m.sum())
        sched.step()
        hist["verse"].append(sums[0] / max(cnts[0], 1))
        hist["urai"].append(sums[1] / max(cnts[1], 1))
        if ep == 1 or ep % 5 == 0:
            print(f"  epoch {ep:2d}/{epochs}  verse-loss={hist['verse'][-1]:.4f}  urai-loss={hist['urai'][-1]:.4f}")
    return hist

joint_ds = JointDataset(train_rows, vocab)
model    = DecoderOnlyLM(len(vocab), pad_id=PAD_ID).to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training on {len(joint_ds)} sequences "
      f"({sum(1 for i in joint_ds.items if i['task']==0)} verse + "
      f"{sum(1 for i in joint_ds.items if i['task']==1)} verse-urai) ...")
hist = train_joint(model, joint_ds, epochs=30)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(hist["verse"]) + 1), hist["verse"], label="verse LM objective")
ax.plot(range(1, len(hist["urai"]) + 1),  hist["urai"],  label="verse->urai objective")
ax.set_xlabel("Epoch"); ax.set_ylabel("Per-token loss")
ax.set_title("Joint Training: Two Objectives, One Decoder")
ax.grid(True, alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

## Test 1 — Does it know the right *words*?

Prompt with the verse, greedily generate a commentary, and compare against the real gold commentary using **token F1** — a bag-of-words overlap score (order doesn't matter, just "how many of the same words appear in both").

**Real result from this run:** mean F1 = **0.161**, median = **0.140**, over 30 verses: low, but the generated text is still fluent, grammatical-*looking* Tamil prose. Reading the actual examples tells the real story: Example 1's generated commentary shares literally **zero words** with the gold text (F1 = 0.000), completely different content, describing something else entirely, while still sounding like plausible commentary prose. Example 5 shows another common failure mode: the exact same short phrase repeating itself three times in a row ("இது உது அறு இறு பொறு அடு விடு பொருள்" ×3). **What this means:** the model has learned the *shape and style* of a commentary: how it opens, its rhythm, its vocabulary, but not reliably the *content* specific to the verse it was actually asked about.

In [ ]:
@torch.no_grad()
def generate(model, prefix_ids, max_new=150):
    model.eval()
    ids = list(prefix_ids)
    for _ in range(max_new):
        window = ids[-(model.max_len - 1):]
        logits = model(torch.tensor([window], dtype=torch.long, device=DEVICE))[0, -1]
        nxt = int(logits.argmax())
        if nxt == EOS_ID: break
        ids.append(nxt)
    return ids

def detok(ids):
    words = []
    for i in ids:
        t = itos.get(int(i), UNK)
        if t in SPECIAL_TOKENS: continue
        else: words.append(t)
    return " ".join(words)

def urai_prompt_ids(row):
    toks = [BOS, VERSE_TAG] + tokenize_words(row["verse"]) + [URAI_TAG]
    return encode_tokens(toks, vocab)

def token_f1(pred_tokens, gold_tokens):
    if not pred_tokens or not gold_tokens: return 0.0
    overlap = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
    if overlap == 0: return 0.0
    p = overlap / len(pred_tokens); r = overlap / len(gold_tokens)
    return 2 * p * r / (p + r)

rng = random.Random(SEED)
eval_rows = rng.sample(heldout_rows, min(30, len(all_rows)))
f1s = []
for i, row in enumerate(tqdm(eval_rows, desc="Generating urai")):
    prompt = urai_prompt_ids(row)
    out    = generate(model, prompt)
    pred_t = [itos[int(x)] for x in out[len(prompt):] if itos[int(x)] not in SPECIAL_TOKENS]
    gold_t = tokenize_words(row["urai"])
    f1s.append(token_f1(pred_t, gold_t))
    if i < 5:
        print("=" * 90)
        print(f"Example {i+1}  [{row['dataset']}]  token-F1={f1s[-1]:.3f}")
        print(f"  VERSE          : {row['verse'][:180]}")
        print(f"  GENERATED URAI : {detok(out[len(prompt):])[:400]}")
        print(f"  GOLD URAI      : {row['urai'][:400]}")

print("=" * 90)
print(f"Urai prediction token-F1 over {len(eval_rows)} verses: "
      f"mean={np.mean(f1s):.3f}  median={np.median(f1s):.3f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(f1s, bins=15, color="tab:blue", alpha=0.85)
ax.axvline(np.mean(f1s), color="tab:red", ls="--", label=f"mean {np.mean(f1s):.3f}")
ax.set_xlabel("Token F1 (generated vs gold urai)"); ax.set_ylabel("Count")
ax.set_title("Test 1: Urai Prediction vs Gold"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Test 2 — Does it know the right *word order*?

For each of the 20 sampled verses, GPT-4.1-mini's clause parse (subject/object/verb/other per clause) drives four different scrambled rewrites of that verse:
- **clause reorder** — shuffle which clause comes first
- **noun swap** — swap the subject and object between clauses
- **verb fronting** — move the verb to the front of its clause
- **sentence reversal** — reverse the whole line order

The trained model then scores the **original** verse and all 4 scrambled versions using its own verse-prediction objective (literally: "how likely does this exact sequence of words look to me"), and a softmax turns those scores into a preference probability across the candidate set. It "wins" an example when the untouched original gets the highest probability of the bunch.

19 of the 20 sampled verses had a usable clause parse with ≥2 clauses and matched back to the training corpus (spanning multiple different source texts this time, not just one) — see the real numbers below.

In [ ]:
OPENAI_PATHS = [
    "data/openai__gpt-4.1-mini.converted.jsonl"
]

def load_openai(paths):
    path = next((p for p in paths if Path(p).exists()), None)
    if path is None:
        print("OpenAI clause file not found in any candidate path.")
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            obj = json.loads(line)
            if obj.get("ok") and obj.get("parsed_json", {}).get("clauses"):
                rows.append({"verse": obj.get("verse", ""),
                             "clauses": obj["parsed_json"]["clauses"]})
    print(f"Loaded {len(rows)} clause parses from {path}")
    return rows

oai_rows     = load_openai(OPENAI_PATHS)
row_by_verse = {norm(r["verse"]): r for r in all_rows}

def match_row(overse):
    r = row_by_verse.get(norm(overse))
    if r is not None:
        return r
    otoks = set(tokenize_words(overse))
    if not otoks:
        return None
    best, best_cov, best_len = None, 0.0, None
    for r in all_rows:
        vtoks = set(tokenize_words(r["verse"]))
        cov = len(otoks & vtoks) / len(otoks)
        if cov > best_cov or (cov == best_cov and best_len is not None and len(vtoks) < best_len):
            best, best_cov, best_len = r, cov, len(vtoks)
    return best if best_cov >= 0.8 else None

eval_pairs = []
for o in oai_rows:
    if len(o["clauses"]) < 2:
        continue
    r = match_row(o["verse"])
    if r is not None:
        # keep the OpenAI verse text (it is what the clauses describe)
        eval_pairs.append({"verse": o["verse"], "clauses": o["clauses"], "dataset": r["dataset"]})
eval_pairs = eval_pairs[:20]
print(f"Matched to corpus (>=2 clauses, capped at 20): {len(eval_pairs)}")
assert eval_pairs, "No OpenAI parses matched the corpus - check data paths above."


In [ ]:
def clause_text(cls):
    return " ".join(c.get("text", "") for c in cls).strip()

def reorder_clauses(cls, rng):
    for _ in range(10):
        c = cls.copy(); rng.shuffle(c)
        if [x.get("text") for x in c] != [x.get("text") for x in cls]:
            return c
    return list(reversed(cls))

def swap_nouns(cls):
    out = [dict(c) for c in cls]
    for i in range(len(out) - 1):
        for field in ("subject", "object"):
            wi = " ".join(out[i].get(field, [])); wj = " ".join(out[i+1].get(field, []))
            if wi and wj and wi in out[i]["text"] and wj in out[i+1]["text"]:
                out[i]["text"]   = out[i]["text"].replace(wi, wj, 1)
                out[i+1]["text"] = out[i+1]["text"].replace(wj, wi, 1)
    return out

def verb_fronting(cls):
    res = []
    for c in cls:
        vs = c.get("verb", [])
        rest = [w for w in c.get("subject", []) + c.get("object", []) + c.get("other", [])
                if w not in vs]
        res.append(dict(c, text=(" ".join(vs) + " " + " ".join(rest)).strip() if vs else c["text"]))
    return res

def scramble_verse(pair):
    """Deterministic per-verse manipulations; identical/empty variants dropped."""
    verse, cls = pair["verse"], pair["clauses"]
    rng   = random.Random(SEED + zlib.crc32(norm(verse).encode("utf-8")))
    sents = [s.strip() for s in re.split(r"[;\n]", verse) if s.strip()]
    variants = {
        "clause_reorder": clause_text(reorder_clauses(cls, rng)),
        "noun_swap":      clause_text(swap_nouns(cls)),
        "verb_fronting":  clause_text(verb_fronting(cls)),
        "sent_reversal":  " ".join(reversed(sents)) if len(sents) > 1 else verse,
    }
    return {k: v for k, v in variants.items() if v.strip() and norm(v) != norm(verse)}

MANIP_TYPES = ["clause_reorder", "noun_swap", "verb_fronting", "sent_reversal"]

In [ ]:
@torch.no_grad()
def verse_logprob(verse, dataset):
    toks = [BOS, VERSE_TAG] + tokenize_words(verse) + [EOS]
    ids  = encode_tokens(toks, vocab)[:model.max_len]
    return sequence_logprob(model, ids, VERSE_PREFIX_LEN)

results = []
for pair in tqdm(eval_pairs, desc="Scoring clause preference"):
    variants = {"original": pair["verse"], **scramble_verse(pair)}
    lp, ntok = {}, {}
    for name, text in variants.items():
        lp[name], ntok[name] = verse_logprob(text, pair["dataset"])
    z     = np.array([lp[n] for n in variants])
    probs = dict(zip(variants, np.exp(z - z.max()) / np.exp(z - z.max()).sum()))
    results.append({"pair": pair, "variants": variants,
                    "logprob": lp, "ntok": ntok, "prob": probs})

SEP = "=" * 90
for i, r in enumerate(results):
    print(SEP)
    print(f"Example {i+1}  [{r['pair']['dataset']}]  ({len(r['variants'])} candidates)")
    print(f"VERSE: {r['pair']['verse'][:160]}")
    print(f"{'variant':16s} {'log P(tokens)':>14s} {'per-token':>10s} {'preference P':>13s}")
    best = max(r["prob"], key=r["prob"].get)
    for name in ["original"] + [m for m in MANIP_TYPES if m in r["variants"]]:
        mark = "  <- preferred" if name == best else ""
        print(f"{name:16s} {r['logprob'][name]:14.2f} {r['logprob'][name]/max(r['ntok'][name],1):10.3f} "
              f"{r['prob'][name]:13.3f}{mark}")
        if name != "original":
            print(f"                 {r['variants'][name][:120]}")
    print(f"  -> Original preferred: {'YES' if best == 'original' else 'NO'}")

print(SEP)
print("SUMMARY\n")
top1 = sum(1 for r in results if max(r["prob"], key=r["prob"].get) == "original")
print(f"Original verse ranked best : {top1}/{len(results)}  ({top1/len(results):.0%})")
print(f"Mean P(original)           : {np.mean([r['prob']['original'] for r in results]):.3f}")
print(f"Mean chance level          : {np.mean([1/len(r['variants']) for r in results]):.3f}\n")
print("Pairwise 'original beats manipulation':")
for mt in MANIP_TYPES:
    wins = sum(1 for r in results if mt in r["prob"] and r["prob"]["original"] > r["prob"][mt])
    tot  = sum(1 for r in results if mt in r["prob"])
    if tot:
        print(f"  vs {mt:16s}: {wins}/{tot}  ({wins/tot:.0%})")

In [ ]:
VARIANT_COLORS = {"original": "tab:blue", "clause_reorder": "tab:orange",
                  "noun_swap": "tab:green", "verb_fronting": "tab:red",
                  "sent_reversal": "tab:purple"}
names = ["original"] + MANIP_TYPES
x = np.arange(len(results)); w = 0.16

fig, ax = plt.subplots(figsize=(max(10, 1.4 * len(results)), 4.5))
for k, name in enumerate(names):
    vals = [r["prob"].get(name, np.nan) for r in results]
    ax.bar(x + k * w, vals, w, label=name, color=VARIANT_COLORS[name], alpha=0.85)
for i, r in enumerate(results):                      # per-example chance level
    ax.hlines(1 / len(r["variants"]), x[i] - w/2, x[i] + w * (len(names) - 0.5),
              colors="gray", ls="--", lw=1)
ax.set_xticks(x + 2 * w)
ax.set_xticklabels([f"Ex {i+1}\n[{r['pair']['dataset'][:8]}]" for i, r in enumerate(results)])
ax.set_ylabel("Preference probability")
ax.set_title("Test 2: Clause Preference — softmax of verse log-probability\n"
             "(dashed line = chance level per example)")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
accs, labels = [], []
for mt in MANIP_TYPES:
    wins = sum(1 for r in results if mt in r["prob"] and r["prob"]["original"] > r["prob"][mt])
    tot  = sum(1 for r in results if mt in r["prob"])
    if tot:
        accs.append(wins / tot); labels.append(f"{mt}\n(n={tot})")
ax.bar(range(len(accs)), accs, color="tab:blue", alpha=0.85)
ax.set_xticks(range(len(accs))); ax.set_xticklabels(labels, fontsize=8)
ax.axhline(0.5, ls="--", c="gray", lw=1, label="chance (pairwise)")
ax.set_ylim(0, 1.05); ax.set_ylabel("P(original) > P(manipulated)")
ax.set_title("Clause preference accuracy per manipulation type")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## Key result: the model strongly prefers real Tamil word order

**Real result from this run — every single comparison, the model picked the real, unscrambled verse:**

| | Result |
|---|---|
| Original verse ranked best | **19 / 19 (100%)** |
| Mean preference probability for the original | **1.000** |
| Mean chance level (if guessing randomly) | 0.211 |

Broken down by scramble type, the original also won **every individual pairwise comparison**:

| vs. | Win rate |
|---|---|
| clause reorder | 19/19 (100%) |
| noun swap | 16/16 (100%) |
| verb fronting | 18/18 (100%) |
| sentence reversal | 19/19 (100%) |

**What this means:** unlike Test 1 (which showed the model doesn't reliably know *what* a specific verse's commentary should say), this test shows it has strongly internalised *how Tamil words are supposed to be ordered* — reordering clauses, swapping subject/object, fronting the verb, or reversing the line all made the text look reliably less "real" to the model, every time.

**The honest caveat, same as always with this test:** these are verses the model was directly trained on (not unseen text), so a clean sweep could partly reflect memorising the exact literal sequence rather than a generalisable grammar rule.

## Test 3 - Shared pairwise grammar probe (comparable to notebook 06)

Tests 1 and 2 above are this notebook's own. **This test exists so the
from-scratch decoder and the fine-tuned Tamil-LLaMA of notebook 06 can be
compared directly**, which Test 2 does not permit:

| | stimuli | design | chance |
|---|---|---|---|
| Test 2 (this notebook) | 4 corruptions generated at runtime from 19 clause parses | 5-way softmax | 0.211 |
| Notebook 06 | pre-generated `probe_*.jsonl` pairs | pairwise | 0.50 |
| **Test 3 (below)** | **same `probe_*.jsonl` pairs** | **pairwise** | **0.50** |

Same files, same rule (`logprob(original) > logprob(perturbed)`), same chance
level - so the two models' accuracies sit on one scale.

**`probe_morphology` is reported separately and excluded from the pooled
figure.** It is the only probe whose perturbation introduces a word absent
from the corpus (`à®¤à¯‹à®´à®¿à®•à¯à®•à¯` -> `à®¤à¯‹à®´à®¿à®‡à®²à¯`). A word-level model maps that to
`<unk>` and wins on vocabulary rather than morphology; the subword model in
notebook 06 has no such problem. Comparing it across the two would be
meaningless. The other 112 items are OOV-clean on both sides.

In [ ]:
from pathlib import Path as _Path

PROBE_DIR = _Path("data/grammar_probe")
_probe_files = sorted(PROBE_DIR.glob("probe_*.jsonl"))
print("probe files:", [p.name for p in _probe_files])

_vocab_words = set(vocab)

def _oov_count(text):
    return sum(1 for w in tokenize_words(text) if w not in _vocab_words)

@torch.no_grad()
def probe_logprob(text):
    """Total log-prob of a verse string, identical prefix handling to Test 2."""
    toks = [BOS, VERSE_TAG] + tokenize_words(text) + [EOS]
    ids  = encode_tokens(toks, vocab)[:model.max_len]
    return sequence_logprob(model, ids, VERSE_PREFIX_LEN)[0]

rows_out, per_probe = [], []
for pf in _probe_files:
    items = [json.loads(l) for l in pf.read_text(encoding="utf-8").splitlines() if l.strip()]
    ptype = pf.stem.replace("probe_", "")
    correct = 0
    oov_items = 0
    for it in items:
        lo, lp = probe_logprob(it["original"]), probe_logprob(it["perturbed"])
        ok = lo > lp
        correct += int(ok)
        oov_items += int(_oov_count(it["perturbed"]) > _oov_count(it["original"]))
        rows_out.append({"probe_type": ptype, "lp_original": lo,
                         "lp_perturbed": lp, "correct": ok})
    per_probe.append({"probe_type": ptype, "n": len(items), "correct": correct,
                      "accuracy": correct / max(len(items), 1),
                      "items_with_oov_perturbation": oov_items})

probe3_df = pd.DataFrame(per_probe)
print()
print(probe3_df.to_string(index=False))

_pooled = [r for r in rows_out if r["probe_type"] != "morphology"]
_n, _c = len(_pooled), sum(r["correct"] for r in _pooled)
print()
print("=" * 62)
print(f"POOLED (morphology excluded): {_c}/{_n} = {_c/_n:.1%}   chance = 50.0%")
print("=" * 62)
print("Report this number against notebook 06's pairwise accuracy.")
print("Report morphology (n=1) separately - see the note above.")

## Use Case Study

Across all models developed in this project—including LSTM and BiLSTM encoders, attention-based encoders, Siamese networks, the mBART-style encoder–decoder, and the decoder-only model—the primary objective has been to determine whether classical Tamil verses and their traditional commentaries can be represented as meaningful vector embeddings for comparison, retrieval, and text generation.

The results indicate that this is feasible, although the problem remains far from solved. The models successfully captured aspects of grammatical structure, such as the decoder-only model's perfect preference for correct word order in Test 2, while still struggling to generate verse-specific content with high lexical accuracy in Test 1. These findings suggest that the learned representations encode meaningful linguistic information despite the complexity of the task.

### Potential Applications

Vectorising classical Tamil literature enables several important applications:

* **Retrieval-Augmented Generation (RAG) and semantic search:** Users can retrieve relevant verses and commentaries based on semantic similarity rather than manual keyword searches.
* **Conversational access:** AI assistants can answer questions about *Tholkappiyam* by grounding responses in the original verses and traditional commentaries.
* **Computational linguistics:** Learned alignments between verses and explanations provide quantitative insights into classical grammar and vocabulary across large corpora.
* **Language change analysis:** Embedding-based comparisons can support corpus-scale studies of lexical and grammatical evolution from Classical to Modern Tamil.

### Future Work

The next stage of this research will focus on improving the representation of Classical Tamil at the tokenisation level, with particular emphasis on preserving linguistically meaningful words and morphemes during subword segmentation. Future work will also explore incorporating these texts into large language models through parameter-efficient and full fine-tuning approaches, enabling models to better understand, retrieve, and generate content grounded in classical Tamil literature and its traditional commentaries.

Overall, this project demonstrates that neural models can learn meaningful structural and semantic representations of classical Tamil, providing a foundation for future research and the development of larger-scale language technologies.
